# 进入目录且导入库

In [1]:
# 进入目录
%cd /root/autodl-tmp/qwen_finetune

/root/autodl-tmp/qwen_finetune


/root/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import GRPOTrainer, GRPOConfig, DPOTrainer, DPOConfig
from datasets import load_dataset, Dataset
from tqdm import tqdm
import pandas as pd

In [47]:
# from vllm import LLM, SamplingParams

ModuleNotFoundError: No module named 'vllm.entrypoints.llm'

In [3]:
# 查看pip缓存大小
!du -sh ~/.cache/pip

# 清空pip缓存（推荐）
!pip cache purge

32K	/root/.cache/pip
Files removed: 0


# 安装依赖

In [67]:
# !pip uninstall torch torchvision torchaudio xformers cuda-python cuda-bindings -y

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.19.0
Uninstalling torchvision-0.19.0:
  Successfully uninstalled torchvision-0.19.0
Found existing installation: torchaudio 2.9.1
Uninstalling torchaudio-2.9.1:
  Successfully uninstalled torchaudio-2.9.1
Found existing installation: xformers 0.0.27.post2
Uninstalling xformers-0.0.27.post2:
  Successfully uninstalled xformers-0.0.27.post2
Found existing installation: cuda-python 13.1.1
Uninstalling cuda-python-13.1.1:
  Successfully uninstalled cuda-python-13.1.1
Found existing installation: cuda-bindings 12.9.4
Uninstalling cuda-bindings-12.9.4:
  Successfully uninstalled cuda-bindings-12.9.4


In [14]:
# 安装依赖
# !pip install -r requirements.txt
# !pip install modelscope
# !pip install vllm

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [43]:
!pip uninstall vllm -y

Found existing installation: vllm 0.6.3.post1
Uninstalling vllm-0.6.3.post1:
  Successfully uninstalled vllm-0.6.3.post1


In [69]:
!python -c "import torch; print(f'CUDA: {torch.version.cuda}, PyTorch: {torch.__version__}')"

CUDA: 12.8, PyTorch: 2.10.0+cu128


In [38]:
!python -c "from vllm import LLM, SamplingParams; print('vLLM 安装成功！')"

vLLM 安装成功！


In [70]:
# 1. 检查 PyTorch 是否还能正常使用
!python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"

# 2. 检查 transformers 是否正常
!python -c "from transformers import AutoModelForCausalLM; print('Transformers OK')"

2.10.0+cu128
True
Transformers OK


# 获取CMMLU数据

In [4]:
OUTPUT_DIR = "/root/autodl-tmp/qwen_finetune"

# 定义要使用的子集
SELECTED_SUBSETS = [
    'computer_science', # 计算机科学 - STEM
    'high_school_mathematics', # 高中数学 - 基础学科
    'chinese_literature', # 中国文学 - 人文学科
    'elementary_commonsense', # 常识 - 基础能力
    'professional_psychology' # 心理学 - 社科类
]

# 检查数据是否已存在
data_dir = os.path.join(OUTPUT_DIR, "data")
save_path = os.path.join(data_dir, "cmmlu.json")

if os.path.exists(save_path):
    print(f"CMMLU 数据已存在: {save_path}")
    loaded_dataset = Dataset.from_json(save_path)
    print(f"已加载数据集，大小: {len(loaded_dataset)}")
else:
    # 创建 data 子目录
    os.makedirs(data_dir, exist_ok=True)

    # 读取测试数据
    cmmlu_data_dir = "/root/autodl-tmp/qwen_finetune/CMMLU/data/test"
    csv_files = [f for f in os.listdir(cmmlu_data_dir) if f.endswith('.csv')]
    
    # 筛选选定的子集
    selected_csv_files = [f for f in csv_files if f.replace('.csv', '') in SELECTED_SUBSETS]
    print(f"选择的子集: {[f.replace('.csv', '') for f in selected_csv_files]}")

    # 加载选定的测试文件
    all_data = []
    for csv_file in selected_csv_files:
        df = pd.read_csv(os.path.join(cmmlu_data_dir, csv_file))
        df['subject'] = csv_file.replace('.csv', '')  # 添加科目信息
        all_data.append(df)

    # 合并数据
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"Total questions: {len(combined_df)}")
    print(f"Subjects: {combined_df['subject'].unique()}")

    # 转换为 Dataset 格式
    dataset = Dataset.from_pandas(combined_df)

    # 使用 pandas 保存（解决 NotImplementedError）
    combined_df.to_json(save_path, orient="records", lines=False, force_ascii=False)
    print(f"数据集已保存到: {save_path}")

    # 验证保存结果
    loaded_dataset = Dataset.from_json(save_path)
    print(f"验证：加载的数据集大小: {len(loaded_dataset)}")

CMMLU 数据已存在: /root/autodl-tmp/qwen_finetune/data/cmmlu.json
已加载数据集，大小: 1002


# 参数配置

## 模型配置

In [5]:
# 模型配置
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

DRIVE_MOUNT_PATH = "/root"
DRIVE_BASE_DIR = os.path.join(DRIVE_MOUNT_PATH, "autodl-tmp", "qwen_finetune")


## 路径配置

In [6]:

# 路径配置
MODEL_PATHS = {
    "baseline": os.path.join(DRIVE_BASE_DIR, "models", "baseline"),
    "sft": os.path.join(DRIVE_BASE_DIR, "models", "sft"),
    "grpo": os.path.join(DRIVE_BASE_DIR, "models", "grpo")
}

RESULT_PATHS = {
    "baseline": os.path.join(DRIVE_BASE_DIR, "results", "baseline"),
    "sft": os.path.join(DRIVE_BASE_DIR, "results", "sft"),
    "grpo": os.path.join(DRIVE_BASE_DIR, "results", "grpo"),
    "final": os.path.join(DRIVE_BASE_DIR, "results", "final_summary.json")
}

DATA_PATHS = {
    "belle": os.path.join(DRIVE_BASE_DIR, "data", "belle.json"),
    "grpo_preference": os.path.join(DRIVE_BASE_DIR, "data", "grpo_preference.json"),
    "eval": os.path.join(DRIVE_BASE_DIR, "data", "belle_eval.json"),
    "preference": os.path.join(DRIVE_BASE_DIR, "data", "belle_preference.json"),
    "preference_data": os.path.join(DRIVE_BASE_DIR, "data", "preference_data.json"),
}

## CMMLU评估配置

In [7]:
# 评估配置
BATCH_SIZE = 4

# CMMLU 评估配置
# 定义要使用的子集
SELECTED_SUBSETS = [
    'computer_science', # 计算机科学 - STEM
    'high_school_mathematics', # 高中数学 - 基础学科
    'chinese_literature', # 中国文学 - 人文学科
    'elementary_commonsense', # 常识 - 基础能力
    'professional_psychology' # 心理学 - 社科类
]
CMMLU_SUBSETS = SELECTED_SUBSETS
CMMLU_DATA_PATH = os.path.join(DRIVE_BASE_DIR, "data", "cmmlu.json")

## LORA配置

In [8]:
# QLoRA 配置
QLORA_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,                 # ✅ 启用 4-bit 量化（核心） 将模型权重从 16-bit 压缩到 4-bit 加载，大幅降低显存占用（Qwen-7B 可降至 ~6GB）
    bnb_4bit_use_double_quant=True,    # ✅ 启用“双重量化”（省显存） 对 4-bit 量化常数再做一次 8-bit 量化，进一步省 0.5~1GB 显存
    bnb_4bit_quant_type="nf4",         # ✅ 使用 Normal Float 4（比 int4 更适合权重） 使用 Normal Float 4（专为神经网络权重设计），比 int4 保留更多信息
    bnb_4bit_compute_dtype=torch.float16  # ✅ 计算时用 float16（平衡速度/精度） 虽然权重是 4-bit，但前向计算时转为 float16（避免 float32 拖慢速度）
)

# LoRA 配置
LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,      # ✅ 因果语言模型（生成任务） 
    inference_mode=False,              # ✅ 训练模式（True 用于推理） 
    r=8,                               # 低秩矩阵维度 LoRA 低秩分解的秩（rank）。值越大，可学习参数越多，但越容易过拟合 ✅ 标准值（常用 8 或 16）
    lora_alpha=16,                     # 缩放因子  控制 LoRA 更新的缩放强度。实际缩放 = alpha / r = 16/8 = 2 ✅ 推荐 ratio=2（即 alpha=2×r）
    lora_dropout=0.1,                  # 防过拟合 在 LoRA 层加 dropout，防过拟合 ✅ 合理（0.05~0.2 均可）
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]  # 注入 LoRA 的模块
)

## SFT训练配置

In [9]:
# 训练配置
TRAIN_CONFIG = {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 5e-7,
    "num_train_epochs": 1,
    "warmup_ratio": 0.1,
    "save_steps": 1000,
    "logging_steps": 100,
    "max_samples": 50
}



## DPO配置

In [10]:
# 偏好对齐数据量
MAX_SAMPLES = 5000

DPO_TRAINING_ARGS = DPOConfig(
    output_dir=MODEL_PATHS["grpo"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-6,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    
    # DPO 核心超参
    beta=0.1,
    
    # 精度与性能
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="paged_adamw_32bit",      # 适合 QLoRA 的优化器
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # 数据处理
    remove_unused_columns=False,
    dataloader_num_workers=2,
    
    # 其他
    report_to="none",               # 不连 wandb/tensorboard
    seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# 函数

## 加载 CMMLU 数据集

In [11]:
def load_cmmlu_data():
    """
    加载 CMMLU 数据集
    """
    print("从指定路径加载 CMMLU 数据集...")

    # 检查文件是否存在
    if not os.path.exists(CMMLU_DATA_PATH):
        raise FileNotFoundError(f"CMMLU 数据集文件不存在: {CMMLU_DATA_PATH}")

    # 首先查看数据结构
    with open(CMMLU_DATA_PATH, "r", encoding="utf-8") as f:
        import json
        dataset = json.load(f)

    print(f"数据类型: {type(dataset)}")
    if isinstance(dataset, list) and len(dataset) > 0:
        print(f"第一条数据: {dataset[0]}")
        print(f"字段名: {dataset[0].keys()}")
    elif isinstance(dataset, dict):
        print(f"数据键: {list(dataset.keys())}")
        # 检查是否有 'data' 或其他键包含实际数据
        if 'data' in dataset and isinstance(dataset['data'], list):
            print(f"data 键中的第一条数据: {dataset['data'][0]}")

    # 按子集分组
    subset_data = {}
    # 处理不同格式的数据
    if isinstance(dataset, list):
        for item in dataset:
            # 尝试多种可能的字段名
            subset = item.get("subject", "")

            # 尝试多种可能的问题字段名
            question = item.get("Question", "")
            
            # 尝试多种可能的答案字段名
            answer = item.get("Answer", "")

            # 确保选项字段存在
            options = []
            if "A" in item and "B" in item and "C" in item and "D" in item:
                options = [item.get("A", ""), item.get("B", ""), item.get("C", ""), item.get("D", "")]

            # 只有当问题和答案都非空时才添加
            if question and answer and options:
                # 重新格式化为标准格式
                formatted_item = {
                    "question": question,
                    "options": options,
                    "answer": answer,
                    "category": subset
                }

                if subset not in subset_data:
                    subset_data[subset] = []
                subset_data[subset].append(formatted_item)
            else:
                print(f"跳过无效数据: {item}, 问题: {question}, 答案: {answer}, 选项: {options}")
    elif isinstance(dataset, dict):
        # 处理字典格式，可能是 {"data": [...]} 或 {"subject1": [...], "subject2": [...]}
        if 'data' in dataset and isinstance(dataset['data'], list):
            # 处理 {"data": [...]} 格式
            for item in dataset['data']:
                subset = item.get("subject", item.get("category", "default"))
                question = item.get("question", "")
                answer = item.get("answer", "")
                
                options = []
                if "A" in item and "B" in item and "C" in item and "D" in item:
                    options = [item.get("A", ""), item.get("B", ""), item.get("C", ""), item.get("D", "")]
                elif "options" in item and isinstance(item["options"], list):
                    options = item["options"]
                
                if question and answer and options:
                    formatted_item = {
                        "question": question,
                        "options": options,
                        "answer": answer,
                        "category": subset
                    }
                    
                    if subset not in subset_data:
                        subset_data[subset] = []
                    subset_data[subset].append(formatted_item)
        else:
            # 处理 {"subject1": [...], "subject2": [...]} 格式
            for subset, items in dataset.items():
                if isinstance(items, list):
                    for item in items:
                        question = item.get("question", "")
                        answer = item.get("answer", "")
                        
                        options = []
                        if "A" in item and "B" in item and "C" in item and "D" in item:
                            options = [item.get("A", ""), item.get("B", ""), item.get("C", ""), item.get("D", "")]
                        elif "options" in item and isinstance(item["options"], list):
                            options = item["options"]
                        
                        if question and answer and options:
                            formatted_item = {
                                "question": question,
                                "options": options,
                                "answer": answer,
                                "category": subset
                            }
                            
                            if subset not in subset_data:
                                subset_data[subset] = []
                            subset_data[subset].append(formatted_item)
    else:
        # 如果是其他格式，抛出异常
        raise ValueError(f"未知的数据格式: {type(dataset)}")

    # 只保留指定的子集
    selected_subset_data = {}
    for subset in CMMLU_SUBSETS:
        if subset in subset_data:
            selected_subset_data[subset] = subset_data[subset]
            print(f"子集 {subset}: {len(selected_subset_data[subset])} 题")

    if not selected_subset_data:
        print("警告：没有找到指定子集的数据")
        print(f"可用的子集: {list(subset_data.keys())}")

    return selected_subset_data

def format_cmmlu_prompt(question, options, subject):
    """
    构造 CMMLU 格式的 prompt
    
    Args:
        question: 问题
        options: 选项列表
        subject: 科目/子集
    
    Returns:
        格式化后的 prompt
    """
    prompt = f"以下是中国关于{subject}的单项选择题，请选出其中的正确答案，仅输出A/B/C/D中的一个选项（仅输出一个大写字母）。\n\n"
    prompt += f"问题：{question}\n"
    for idx, opt in enumerate(options):
        prompt += f"{chr(65 + idx)}. {opt}\n"
    prompt += "答案："
    return prompt


## 评估CMMLU

In [12]:
def evaluate_model(model_path, result_path, model_name, model, tokenizer, limit=None):
    cmmlu_result_path = os.path.join(result_path, "cmmlu.json")
    cmmlu_details_path = os.path.join(result_path, "cmmlu_details.json")

    # 加载 CMMLU 数据集
    print("加载 CMMLU 数据集...")
    subset_data = load_cmmlu_data()
    
    # 评估结果
    eval_results = {
        "results": {},
        "versions": {"cmmlu_native": "1.0"},
        "config": {}
    }
    
    # 详细评估过程
    evaluation_details = {
        "model_name": model_name,
        "details": {}
    }
    
    total_correct = 0
    total_samples = 0
    
    # 遍历每个子集
    for subset, items in subset_data.items():
        print(f"\n评估子集: {subset}")
        
        # 初始化该子集的详细记录
        evaluation_details["details"][subset] = []
        
        # 限制样本数（如果指定了 limit）
        if limit is not None:
            items = items[:limit]
        
        correct = 0
        samples = len(items)
        
        # 遍历每个问题
        for i, item in enumerate(items):
            # 构造 prompt
            question = item.get("question", "")
            options = item.get("options", [])
            answer = item.get("answer", "")
            
            # 构造 CMMLU 格式的 prompt
            prompt = format_cmmlu_prompt(question, options, subset)
            
            # 构建聊天模板
            messages = [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
            
            # 生成回答
            # 应用聊天模板
            chat_input = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            )

            # 修正：正确提取 input_ids tensor
            input_ids = chat_input['input_ids'].to(model.device)  # ✅ 提取 tensor 再移动设备

            # 生成回答
            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    max_new_tokens=10,
                    temperature=0.0,
                    top_p=1.0,
                    do_sample=False
                )

            # 解码输出
            output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
            
            # 提取答案
            predicted_answer = ""
            # 尝试从输出中提取 A/B/C/D
            for char in output:
                if char in ["A", "B", "C", "D"]:
                    predicted_answer = char
                    break
            
            # 检查答案
            is_correct = predicted_answer == answer
            if is_correct:
                correct += 1
            
            # 记录详细信息
            evaluation_details["details"][subset].append({
                "question_id": i + 1,
                "question": question,
                "options": options,
                "prompt": prompt,
                "model_output": output,
                "predicted_answer": predicted_answer,
                "correct_answer": answer,
                "is_correct": is_correct
            })
            
            # 进度反馈
            if (i + 1) % 100 == 0:
                print(f"  已评估 {i + 1}/{samples} 题，正确率: {correct / (i + 1):.4f}")
                    
        
        # 计算子集准确率
        accuracy = correct / samples if samples > 0 else 0.0
        eval_results["results"][f"cmmlu_{subset}"] = {
            "acc": accuracy,
            "samples": samples
        }
        
        # 累计总正确数和总样本数
        total_correct += correct
        total_samples += samples
        
        print(f"  子集 {subset} 评估完成，正确率: {accuracy:.4f} ({correct}/{samples})")
    
    # 计算总体准确率
    overall_accuracy = total_correct / total_samples if total_samples > 0 else 0.0
    eval_results["results"]["cmmlu"] = {
        "acc": overall_accuracy,
        "samples": total_samples
    }
    
    # 为了与原有格式兼容，添加 cmmlu:{subset} 格式的结果
    for subset in CMMLU_SUBSETS:
        if f"cmmlu_{subset}" in eval_results["results"]:
            eval_results["results"][f"cmmlu:{subset}"] = {
                "acc": eval_results["results"][f"cmmlu_{subset}"]["acc"],
                "acc_stderr": 0.0
            }
        else:
            eval_results["results"][f"cmmlu:{subset}"] = {
                "acc": 0.0,
                "acc_stderr": 0.0
            }
    
    print(f"\n总体评估完成，总正确率: {overall_accuracy:.4f} ({total_correct}/{total_samples})")
    results = eval_results
        
    # 保存结果
    os.makedirs(result_path, exist_ok=True)
    with open(cmmlu_result_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"{model_name} 评估结果已保存到: {cmmlu_result_path}")
    
    # 保存详细评估过程
    with open(cmmlu_details_path, "w", encoding="utf-8") as f:
        json.dump(evaluation_details, f, ensure_ascii=False, indent=2)
    print(f"{model_name} 详细评估过程已保存到: {cmmlu_details_path}")
    
    return results


## 获取SFT数据

In [13]:
def get_sft_data():
    # 加载并准备数据
    with open(DATA_PATHS["belle"], 'r', encoding='utf-8') as f:
        belle_data = json.load(f)  # 假设是列表格式 [{"instruction": "...", "input": "...", "output": "..."}, ...]
    
    # 转换为 Dataset 格式
    dataset = Dataset.from_list(belle_data)

    # 格式化数据
    def format_example(example):
        # Qwen2.5 模型的输入格式
        messages = [
            {
                "role": "user",
                "content": example["instruction"]
            },
            {
                "role": "assistant",
                "content": example["output"]
            }
        ]
        return {"messages": messages}
    
    # 应用格式化函数
    formatted_dataset = dataset.map(format_example, remove_columns=dataset.column_names)
    
    # 过滤掉空的输入或输出
    formatted_dataset = formatted_dataset.filter(
        lambda x: x["messages"][0]["content"] and x["messages"][1]["content"]
    )
    
    print(f"格式化后的数据量: {len(formatted_dataset)}")

    return formatted_dataset

## SFT训练

In [14]:
def train_sft():
    """
    训练 SFT 模型
    """
    sft_path = MODEL_PATHS["sft"]
    if os.path.exists(os.path.join(sft_path, "config.json")):
        print(f"SFT 模型已存在，路径: {sft_path}")
        return sft_path
    else:
        print("SFT 模型不存在，开始训练...")
        
        # 加载 tokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATHS["baseline"], trust_remote_code=True)
        
        # 加载模型（4-bit 量化）
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATHS["baseline"],
            quantization_config=QLORA_CONFIG,
            device_map="auto",
            trust_remote_code=True
        )
        
        # 添加 LoRA 适配器
        model = get_peft_model(model, LORA_CONFIG)
        print("模型参数:")
        model.print_trainable_parameters()
        
        dataset = get_sft_data()
        
        # 数据预处理函数
        def preprocess_function(example):
            # 按照 Qwen2.5 的格式构建输入
            batch_encoding = tokenizer.apply_chat_template(
                example["messages"],
                add_generation_prompt=False,
                return_tensors="pt"
            )
            # 从 BatchEncoding 中提取 input_ids
            input_ids = batch_encoding.input_ids.squeeze(0)
            return {"input_ids": input_ids, "labels": input_ids}
        
        # 应用预处理函数
        tokenized_dataset = dataset.map(
            preprocess_function,
            batched=False,
            remove_columns=dataset.column_names
        )
        
        # 训练参数
        from transformers import Trainer, TrainingArguments
        training_args = TrainingArguments(
            output_dir=sft_path,
            per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
            gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
            learning_rate=TRAIN_CONFIG["learning_rate"],
            num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
            warmup_ratio=TRAIN_CONFIG["warmup_ratio"],
            save_strategy="steps",
            save_steps=TRAIN_CONFIG["save_steps"],
            logging_steps=TRAIN_CONFIG["logging_steps"],
            bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
            fp16=not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8,
            optim="paged_adamw_32bit",
            lr_scheduler_type="cosine",
            report_to="none"
        )
        
        # 创建 Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset,
            data_collator=lambda data: {
                "input_ids": torch.nn.utils.rnn.pad_sequence(
                    [torch.tensor(item["input_ids"]) for item in data],
                    batch_first=True,
                    padding_value=tokenizer.pad_token_id
                ),
                "labels": torch.nn.utils.rnn.pad_sequence(
                    [torch.tensor(item["labels"]) for item in data],
                    batch_first=True,
                    padding_value=-100
                ),
            }
        )
        
        # 开始训练
        print("开始 SFT 训练...")
        trainer.train()
        
        # 保存模型
        print("保存 SFT 模型...")
        model.save_pretrained(sft_path)
        tokenizer.save_pretrained(sft_path)
        print(f"SFT 模型已保存到: {sft_path}")
        
        # 释放显存
        del model, trainer
        torch.cuda.empty_cache()
        
        return sft_path


## 输出BELLE

### 普通版本

In [15]:
def generate_responses(
    data_path,
    tokenizer,
    model,
    save_path,
    max_new_tokens=512,
    device="cuda"
):
    """
    使用给定的 tokenizer 和 model 对 BELLE 评估数据生成完整回答。
    
    Args:
        data_path (str): BELLE eval JSON 文件路径，格式为 [{"instruction": "...", "input": "...", "output": "..."}, ...]
        tokenizer: Hugging Face tokenizer（需支持 apply_chat_template）
        model: Hugging Face causal language model（已加载到 device）
        save_path (str): 生成结果保存路径（.json）
        max_new_tokens (int): 最大生成长度，默认 512（足够完整回答）
        device (str): 推理设备，默认 "cuda"
    """
    # 加载评估数据
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    results = []

    model.eval()  # 确保在 eval 模式
    with torch.no_grad():
        for item in tqdm(eval_data, desc="Generating responses"):
            # 构建 messages（BELLE 是单轮）
            messages = [
                {"role": "user", "content": item["instruction"] + ("\n" + item["input"] if item["input"].strip() else "")}
            ]
            
            # 使用 tokenizer 的 chat template 构建 prompt
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True  # 添加 assistant 开始标记
            )
            
            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=2048  # 防止超长输入
            ).to(device)
            
            # 生成
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,      # 贪心解码，确保可复现
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            # 解码生成部分（跳过 prompt）
            generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True)
            
            # 保存结果（保留原字段 + 新增）
            result_item = {
                "instruction": item["instruction"],
                "input": item["input"],
                "ground_truth": item.get("output", ""),  # 原始答案（可选）
                "response": response.strip()
            }
            results.append(result_item)
    
    # 保存结果
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 生成完成！共 {len(results)} 条，已保存至: {save_path}")

### transformer优化版本

In [51]:
def generate_responses_optimized(
    data_path,
    model_path,
    save_path,
    max_new_tokens=2048,
    use_tqdm=True
):
    """
    使用优化的 transformers 推理，速度接近 vLLM
    """
    # 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 加载模型（使用最好的加速设置）
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",  # 自动分配到所有设备
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,  # 节省内存
        attn_implementation="flash_attention_2" if torch.cuda.is_available() and hasattr(torch.nn.functional, 'scaled_dot_product_attention') else "eager"
    )
    
    # 设置为评估模式
    model.eval()
    
    # 加载数据
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    results = []
    
    with torch.no_grad():  # 禁用梯度计算
        for i, item in enumerate(tqdm(eval_data, desc="Generating responses")):
            try:
                # 构建 prompt
                content = item["instruction"] + ("\n" + item["input"] if item["input"].strip() else "")
                messages = [{"role": "user", "content": content}]
                
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                
                # Tokenize
                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    padding=False,
                    truncation=True,
                    max_length=3072  # 增加最大长度
                ).to(model.device)
                
                # 生成（使用最快的参数）
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,           # 贪心解码，最快
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    temperature=0.0,           # 确保确定性
                    num_return_sequences=1,    # 只生成一个
                    use_cache=True           # 使用 kv cache 加速
                )
                
                # 解码生成部分
                generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
                response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                
                # 保存结果
                result_item = {
                    "instruction": item["instruction"],
                    "input": item["input"],
                    "ground_truth": item.get("output", ""),
                    "response": response
                }
                results.append(result_item)
                
                # 清理缓存（防止显存堆积）
                if i % 10 == 0:  # 每10条清理一次
                    torch.cuda.empty_cache()
                    gc.collect()
                    
            except Exception as e:
                print(f"Error processing item {i}: {e}")
                # 添加错误占位符
                result_item = {
                    "instruction": item["instruction"],
                    "input": item["input"],
                    "ground_truth": item.get("output", ""),
                    "response": f"[ERROR: {str(e)}]"
                }
                results.append(result_item)
    
    # 保存结果
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 生成完成！共 {len(results)} 条，已保存至: {save_path}")
    
    # 清理
    del model
    torch.cuda.empty_cache()

### vllm版本

In [40]:
def generate_responses_vllm(
    data_path,
    model_path,
    save_path,
    max_new_tokens=2048,
    gpu_memory_utilization=0.8,  # 显存使用率
    dtype="bfloat16"
):
    """
    使用 vLLM 加速生成回答
    
    Args:
        data_path (str): BELLE eval JSON 文件路径
        model_path (str): 模型路径
        save_path (str): 保存路径
        max_new_tokens (int): 最大生成长度
        gpu_memory_utilization (float): GPU 显存使用率
        dtype (str): 计算精度
    """
    # 加载评估数据
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    # 初始化 vLLM
    llm = LLM(
        model=model_path,
        tensor_parallel_size=1,  # 单卡
        dtype=dtype,
        gpu_memory_utilization=gpu_memory_utilization,
        trust_remote_code=True
    )
    
    # 构建 prompts
    prompts = []
    for item in eval_data:
        messages = [
            {"role": "user", "content": item["instruction"] + ("\n" + item["input"] if item["input"].strip() else "")}
        ]
        prompt = llm.llm_engine.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(prompt)
    
    # 设置采样参数
    sampling_params = SamplingParams(
        max_tokens=max_new_tokens,
        temperature=0.0,  # 贪心解码
        stop_token_ids=[llm.llm_engine.tokenizer.eos_token_id]
    )
    
    # 批量生成
    outputs = llm.generate(prompts, sampling_params)
    
    # 解析结果
    results = []
    for i, output in enumerate(outputs):
        original_item = eval_data[i]
        response = output.outputs[0].text.strip()
        
        result_item = {
            "instruction": original_item["instruction"],
            "input": original_item["input"],
            "ground_truth": original_item.get("output", ""),
            "baseline_response": response
        }
        results.append(result_item)
    
    # 保存结果
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ vLLM 生成完成！共 {len(results)} 条，已保存至: {save_path}")

### 偏好数据

In [16]:
def generate_multiple_responses(
    data_path,
    tokenizer,
    model,
    save_path,
    num_samples=4,           # 每条指令生成多少个回答
    max_new_tokens=2048,
    device="cuda",
    temperature=0.7,         # 控制多样性（>0 才有多样性）
    top_p=0.9,
    seed=42                  # 可复现
):
    """
    构造偏好数据：对每条指令生成多个不同回答。
    
    输出格式：
    [
      {
        "instruction": "...",
        "input": "...",
        "ground_truth": "...",
        "responses": ["resp1", "resp2", "resp3", "resp4"]  # 多个回答
      },
      ...
    ]
    """
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 加载前 N 条数据（这里取全部，你可以在调用时传入切片）
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    results = []
    model.eval()

    with torch.no_grad():
        for item in tqdm(eval_data, desc=f"Generating {num_samples} responses per prompt"):
            messages = [
                {"role": "user", "content": item["instruction"] + ("\n" + item["input"] if item["input"].strip() else "")}
            ]
            
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=2048
            ).to(device)
            
            responses = []
            for i in range(num_samples):
                # 关键：开启 do_sample 并设置 temperature/top_p
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,          # 必须为 True 才有多样性
                    temperature=temperature,
                    top_p=top_p,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    # 防止重复
                    repetition_penalty=1.1
                )
                
                generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
                response = tokenizer.decode(generated_ids, skip_special_tokens=True)
                responses.append(response.strip())
            
            result_item = {
                "instruction": item["instruction"],
                "input": item["input"],
                "ground_truth": item.get("output", ""),
                "responses": responses  # 注意这里是列表
            }
            results.append(result_item)
    
    # 保存
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 候选回答生成完成！共 {len(results)} 条指令，每条 {num_samples} 个回答")
    print(f"📁 已保存至: {save_path}")

# Baseline

In [10]:
# 加载 tokenizer
tokenizer_baseline = AutoTokenizer.from_pretrained(MODEL_PATHS["baseline"], trust_remote_code=True)

# 加载模型
model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_PATHS["baseline"],
    quantization_config=QLORA_CONFIG,
    device_map="auto",
    trust_remote_code=True
)

print("模型加载成功！")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

模型加载成功！


In [11]:
# 阶段 3：基线模型CMMLU评估
print("\n阶段 3：基线模型CMMLU评估")
baseline_results = evaluate_model(MODEL_PATHS["baseline"], RESULT_PATHS["baseline"], "基线", model_baseline, tokenizer_baseline, limit=100)

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



阶段 3：基线模型CMMLU评估
加载 CMMLU 数据集...
从指定路径加载 CMMLU 数据集...
数据类型: <class 'list'>
第一条数据: {'Unnamed: 0': 0, 'Question': '《日出》的结构采用的是', 'A': '散点透视法', 'B': '回溯法', 'C': '拴桩法', 'D': '心理分析法', 'Answer': 'A', 'subject': 'chinese_literature'}
字段名: dict_keys(['Unnamed: 0', 'Question', 'A', 'B', 'C', 'D', 'Answer', 'subject'])
子集 computer_science: 204 题
子集 high_school_mathematics: 164 题
子集 chinese_literature: 204 题
子集 elementary_commonsense: 198 题
子集 professional_psychology: 232 题

评估子集: computer_science
  已评估 100/100 题，正确率: 0.6500
  子集 computer_science 评估完成，正确率: 0.6500 (65/100)

评估子集: high_school_mathematics
  已评估 100/100 题，正确率: 0.2300
  子集 high_school_mathematics 评估完成，正确率: 0.2300 (23/100)

评估子集: chinese_literature
  已评估 100/100 题，正确率: 0.5000
  子集 chinese_literature 评估完成，正确率: 0.5000 (50/100)

评估子集: elementary_commonsense
  已评估 100/100 题，正确率: 0.6500
  子集 elementary_commonsense 评估完成，正确率: 0.6500 (65/100)

评估子集: professional_psychology
  已评估 100/100 题，正确率: 0.7200
  子集 professional_psychology 评估完成，正确率: 0.7

In [65]:
# 调用函数
generate_responses(
    data_path=DATA_PATHS["eval"],
    tokenizer=tokenizer_baseline,
    model=model_baseline,
    save_path=os.path.join(RESULT_PATHS["baseline"], "eval_response.json"),
    max_new_tokens=2048
)

Generating responses:   0%|          | 0/500 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [41]:
# vllm推理加速
generate_responses_vllm(
    data_path=DATA_PATHS["eval"],
    model_path=MODEL_PATHS["baseline"],
    save_path=os.path.join(RESULT_PATHS["baseline"], "eval_response.json"),
    max_new_tokens=2048
)

NameError: name 'LLM' is not defined

# SFT

## 训练

In [18]:
# 阶段 4：SFT 训练
print("\n阶段 4：SFT 训练")
sft_path = train_sft()


阶段 4：SFT 训练
SFT 模型不存在，开始训练...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

模型参数:
trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


开始 SFT 训练...


Step,Training Loss
100,2.125122
200,2.120550
300,2.086945
400,1.973178
500,1.942058
600,1.870631
700,1.840461
800,1.766505
900,1.749787
1000,1.709610


保存 SFT 模型...
SFT 模型已保存到: /root/autodl-tmp/qwen_finetune/models/sft


## 评估CMMLU+生成response

In [16]:
model_path = MODEL_PATHS["sft"]
result_path = RESULT_PATHS["sft"]
model_name = 'SFT'

# 加载 tokenizer
tokenizer_sft = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 检查是否是已经合并的模型还是带 LoRA 的模型
adapter_config_path = os.path.join(model_path, "adapter_config.json")

if os.path.exists(adapter_config_path):
    # 如果存在 adapter_config.json，说明是 LoRA 模型
    print("检测到 LoRA 模型，正在加载...")
    
    # 先加载基础模型（不使用量化）
    model_sft = AutoModelForCausalLM.from_pretrained(
        MODEL_PATHS["baseline"],  # 从基础模型加载
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    
    # 加载 LoRA 权重并合并
    print(f"加载 {model_name} 模型的 LoRA 权重并合并...")
    model_sft = PeftModel.from_pretrained(model_sft, model_path)
    model_sft = model_sft.merge_and_unload()
    print(f"{model_name} 模型 LoRA 权重合并完成")
else:
    # 如果没有 adapter_config.json，说明已经是合并后的模型
    print("检测到已合并模型，直接加载...")
    model_sft = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,  # 移除量化配置，因为已经是合并模型
        device_map="auto",
        trust_remote_code=True
    )

`torch_dtype` is deprecated! Use `dtype` instead!


检测到 LoRA 模型，正在加载...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

加载 SFT 模型的 LoRA 权重并合并...
SFT 模型 LoRA 权重合并完成


In [20]:
# 阶段 5：SFT模型评估
print("\n阶段 5：SFT模型评估")
sft_results = evaluate_model(MODEL_PATHS["sft"], RESULT_PATHS["sft"], "SFT", model_sft, tokenizer_sft, limit=100)

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



阶段 5：SFT模型评估
加载 CMMLU 数据集...
从指定路径加载 CMMLU 数据集...
数据类型: <class 'list'>
第一条数据: {'Unnamed: 0': 0, 'Question': '《日出》的结构采用的是', 'A': '散点透视法', 'B': '回溯法', 'C': '拴桩法', 'D': '心理分析法', 'Answer': 'A', 'subject': 'chinese_literature'}
字段名: dict_keys(['Unnamed: 0', 'Question', 'A', 'B', 'C', 'D', 'Answer', 'subject'])
子集 computer_science: 204 题
子集 high_school_mathematics: 164 题
子集 chinese_literature: 204 题
子集 elementary_commonsense: 198 题
子集 professional_psychology: 232 题

评估子集: computer_science
  已评估 100/100 题，正确率: 0.7000
  子集 computer_science 评估完成，正确率: 0.7000 (70/100)

评估子集: high_school_mathematics
  已评估 100/100 题，正确率: 0.3600
  子集 high_school_mathematics 评估完成，正确率: 0.3600 (36/100)

评估子集: chinese_literature
  已评估 100/100 题，正确率: 0.5300
  子集 chinese_literature 评估完成，正确率: 0.5300 (53/100)

评估子集: elementary_commonsense
  已评估 100/100 题，正确率: 0.7000
  子集 elementary_commonsense 评估完成，正确率: 0.7000 (70/100)

评估子集: professional_psychology
  已评估 100/100 题，正确率: 0.6700
  子集 professional_psychology 评估完成，正确率: 0.6700 

In [21]:
# 调用函数
generate_responses(
    data_path=DATA_PATHS["eval"],
    tokenizer=tokenizer_sft,
    model=model_sft,
    save_path=os.path.join(RESULT_PATHS["sft"], "eval_response.json"),
    max_new_tokens=2048
)

Generating responses: 100%|██████████| 500/500 [11:44<00:00,  1.41s/it]

✅ 生成完成！共 500 条，已保存至: /root/autodl-tmp/qwen_finetune/results/sft/eval_response.json


## 生成人类偏好对齐的候选数据

In [87]:
# 路径配置
preference_data_path = DATA_PATHS["preference"]  # 你的 BELLE 评估集
grpo_output_path = os.path.join(DRIVE_BASE_DIR, "data", "belle_preference_response.json")

# 只取前 若干 条（可调）
with open(preference_data_path, "r", encoding="utf-8") as f:
    all_data = json.load(f)
    first = all_data[:]

temp_file = "/tmp/belle_first.json"
with open(temp_file, "w", encoding="utf-8") as f:
    json.dump(first, f, ensure_ascii=False, indent=2)

# 生成多回答
generate_multiple_responses(
    data_path=temp_file,
    tokenizer=tokenizer_sft,
    model=model_sft,
    save_path=grpo_output_path,
    num_samples=4,          # 每条生成 4 个回答
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.9,
    device="cuda"
)

Generating 4 responses per prompt:   2%|▏         | 150/10000 [11:55<13:02:43,  4.77s/it]


KeyboardInterrupt: 

# 人类偏好对齐

## 函数

### 加载并清洗偏好数据

In [17]:
def load_preference_data():
    """
    加载并清洗偏好数据
    """
    data_path = DATA_PATHS['preference_data']
    print(f"加载偏好数据: {data_path}")
    
    # 检查文件是否存在
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"偏好数据文件不存在: {data_path}")
    
    # 加载数据
    with open(data_path, "r", encoding="utf-8") as f:
        preference_data = json.load(f)
    
    print(f"原始数据量: {len(preference_data)}")
    
    # 数据清洗
    cleaned_data = []
    for item in preference_data:
        # 过滤掉 sorted_indices is None 的样本
        if "sorted_indices" not in item or item["sorted_indices"] is None:
            continue
        
        # 过滤掉 responses 中存在任意两个完全相同回答的样本
        responses = item.get("responses", [])
        if len(responses) != len(set(responses)):
            continue
        
        # 确保有足够的回答
        if len(responses) < 2:
            continue
        
        cleaned_data.append(item)
    
    print(f"清洗后数据量: {len(cleaned_data)}")
    
    # 取前 N 条数据
    if MAX_SAMPLES is not None and len(cleaned_data) > MAX_SAMPLES:
        cleaned_data = cleaned_data[:MAX_SAMPLES]
        print(f"限制数据量为: {MAX_SAMPLES} 条")
    
    return cleaned_data

### 从偏好数据构建 DPO 数据集

In [18]:
def build_dpo_dataset(preference_data, tokenizer):
    """
    从偏好数据构建 DPO 数据集（使用 Qwen 官方 chat template）
    
    Args:
        preference_data: 原始偏好数据列表
        tokenizer: 已加载的 Qwen tokenizer（需支持 apply_chat_template）
    """
    dpo_data = []
    
    for item in preference_data:
        instruction = item.get("instruction", "").strip()
        responses = [r.strip() for r in item.get("responses", [])]
        sorted_indices = item.get("sorted_indices", [])
        
        if len(sorted_indices) >= 2:
            chosen_idx = sorted_indices[0]
            rejected_idx = sorted_indices[-1]
            
            if (0 <= chosen_idx < len(responses)) and (0 <= rejected_idx < len(responses)):
                chosen = responses[chosen_idx]
                rejected = responses[rejected_idx]
                
                # ✅ 关键修改：用 apply_chat_template 生成标准 prompt
                messages = [{"role": "user", "content": instruction}]
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True  # 包含 <|im_start|>assistant\n
                )
                
                dpo_item = {
                    "prompt": prompt,
                    "chosen": chosen,
                    "rejected": rejected
                }
                dpo_data.append(dpo_item)

    print(f"DPO 数据集大小: {len(dpo_data)}")
    
    # 转换为 Dataset 对象
    dataset = Dataset.from_list(dpo_data)
    return dataset

### DPO训练

In [19]:
def train_dpo(tokenizer, base_model, dpo_dataset):
    """
    训练 DPO 模型
    - base_model: 可以是已合并的完整模型（如 SFT 合并后）
    - 函数内部会重新添加 LoRA 适配器用于 DPO 微调
    """
    # 1. 给 base_model 重新加上 LoRA（用于 DPO 微调）
    print("为模型添加 LoRA 适配器用于 DPO...")
    model_for_dpo = get_peft_model(base_model, LORA_CONFIG)
    
    # 2. 创建 DPO Trainer
    dpo_trainer = DPOTrainer(
        model=model_for_dpo,
        args=DPO_TRAINING_ARGS,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )
    
    print("开始 DPO 训练...")
    dpo_trainer.train()
    print("DPO 训练完成")
    
    # 3. 合并 LoRA 权重（现在 model_for_dpo 是 PeftModel，有 merge_and_unload）
    print("合并 DPO 的 LoRA 权重...")
    merged_model = dpo_trainer.model.merge_and_unload()
    
    # 4. 保存模型和 tokenizer
    save_path = MODEL_PATHS["grpo"]
    os.makedirs(save_path, exist_ok=True)
    merged_model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"DPO 模型已保存到: {save_path}")
    
    # 5. 清理显存
    del model_for_dpo, dpo_trainer
    torch.cuda.empty_cache()
    print("显存已释放")
    
    return save_path, merged_model, tokenizer

## DPO

In [21]:
# 加载并处理偏好数据
preference_data = load_preference_data()

加载偏好数据: /root/autodl-tmp/qwen_finetune/data/preference_data.json
原始数据量: 8836
清洗后数据量: 5035
限制数据量为: 5000 条


In [22]:
# 构造适用于DPO的数据集
dpo_dataset = build_dpo_dataset(preference_data, tokenizer_sft)

DPO 数据集大小: 5000


In [36]:
# 训练 DPO 模型
dpo_model_path, model_dpo, tokenizer_dpo = train_dpo(tokenizer_sft, model_sft, dpo_dataset)

为模型添加 LoRA 适配器用于 DPO...


/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

开始 DPO 训练...


Step,Training Loss
10,0.694179
20,0.693920
30,0.692734
40,0.694644
50,0.693116
60,0.689625
70,0.691294
80,0.689632
90,0.686650
100,0.689595


DPO 训练完成
合并 DPO 的 LoRA 权重...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DPO 模型已保存到: /root/autodl-tmp/qwen_finetune/models/grpo
显存已释放


In [20]:
# DPO 模型路径
model_path = MODEL_PATHS["grpo"]
result_path = RESULT_PATHS["grpo"]
model_name = 'DPO'

# 加载 tokenizer
tokenizer_dpo = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 检查是否是 LoRA 模型（虽然你训练时已合并，但为了健壮性保留判断）
adapter_config_path = os.path.join(model_path, "adapter_config.json")

if os.path.exists(adapter_config_path):
    # 理论上不会进入这里，除非你改了 train_dpo 不合并
    print("检测到 DPO LoRA 模型，正在加载并合并...")
    
    # 从 baseline 加载基础模型（注意：DPO 是在 SFT 上微调的，所以基础模型应是 SFT 合并后的模型！）
    base_model_for_dpo = AutoModelForCausalLM.from_pretrained(
        MODEL_PATHS["sft"],  # ← 关键：DPO 的 base 是 SFT 模型，不是原始 baseline！
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    
    model_dpo = PeftModel.from_pretrained(base_model_for_dpo, model_path)
    model_dpo = model_dpo.merge_and_unload()
    print("DPO LoRA 权重合并完成")
else:
    # 正常情况：你已保存合并模型，直接加载
    print("检测到已合并的 DPO 模型，直接加载...")
    model_dpo = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

print(f"✅ {model_name} 模型和 tokenizer 加载成功！")

`torch_dtype` is deprecated! Use `dtype` instead!


检测到已合并的 DPO 模型，直接加载...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ DPO 模型和 tokenizer 加载成功！


In [37]:
# 阶段 7：DPO模型评估
print("\n阶段 7：DPO模型评估")
dpo_results = evaluate_model(MODEL_PATHS["grpo"], RESULT_PATHS["grpo"], "DPO", model_dpo, tokenizer_dpo, limit=100)


阶段 7：DPO模型评估
加载 CMMLU 数据集...
从指定路径加载 CMMLU 数据集...
数据类型: <class 'list'>
第一条数据: {'Unnamed: 0': 0, 'Question': '《日出》的结构采用的是', 'A': '散点透视法', 'B': '回溯法', 'C': '拴桩法', 'D': '心理分析法', 'Answer': 'A', 'subject': 'chinese_literature'}
字段名: dict_keys(['Unnamed: 0', 'Question', 'A', 'B', 'C', 'D', 'Answer', 'subject'])
子集 computer_science: 204 题
子集 high_school_mathematics: 164 题
子集 chinese_literature: 204 题
子集 elementary_commonsense: 198 题
子集 professional_psychology: 232 题

评估子集: computer_science
  已评估 100/100 题，正确率: 0.7000
  子集 computer_science 评估完成，正确率: 0.7000 (70/100)

评估子集: high_school_mathematics
  已评估 100/100 题，正确率: 0.1700
  子集 high_school_mathematics 评估完成，正确率: 0.1700 (17/100)

评估子集: chinese_literature
  已评估 100/100 题，正确率: 0.5000
  子集 chinese_literature 评估完成，正确率: 0.5000 (50/100)

评估子集: elementary_commonsense
  已评估 100/100 题，正确率: 0.7000
  子集 elementary_commonsense 评估完成，正确率: 0.7000 (70/100)

评估子集: professional_psychology
  已评估 100/100 题，正确率: 0.7100
  子集 professional_psychology 评估完成，正确率: 0.7100 

In [21]:
# 调用函数
generate_responses(
    data_path=DATA_PATHS["eval"],
    tokenizer=tokenizer_dpo,
    model=model_dpo,
    save_path=os.path.join(RESULT_PATHS["grpo"], "eval_response.json"),
    max_new_tokens=2048
)

Generating responses: 100%|██████████| 500/500 [23:04<00:00,  2.77s/it]

✅ 生成完成！共 500 条，已保存至: /root/autodl-tmp/qwen_finetune/results/grpo/eval_response.json
